# 07. Strands Decider 체험 — Strands 팀의 전용 decision model

[strands-decider](https://github.com/strands-labs/strands-decider)는 Strands Agents 팀이 만든 **전용 decision model**입니다 (Apache-2.0, HF `StrandsAgents` org — `amazon` org에도 미러됨).

- 구조: 사전학습 LLM torso(Qwen3.5-2B-Base 또는 Gemma 4)에서 **LM head를 떼고 ~1M 파라미터 pointer head**를 얹은 뒤 LoRA로 적응 — 생성 없이 한 번의 forward pass로 답합니다.
- 질문 유형: noul(yes/no 확률), choice(N지선다), score(순서형 척도) + **calibrated confidence** (README 기준: 처음 보는 분류 태스크에서 confidence 0.9 이상 답의 정답률 약 95%).
- 서버 모드는 **`POST /v1/systemone`** (Jev/System One wire format) — 지금까지 쓰던 SGLang과 **같은 형식**이라, 우리 workflow에 client 그대로 꽂을 수 있습니다.
- Strands Agents SDK와의 공식 통합: `InterventionHandler`로 tool 호출 직전 "인자가 근거 있는가 / 성급한 호출인가"를 decider에게 물어 Proceed/Guide를 결정하는 예제가 제공됩니다.

### 서버 기동 (이 EC2에는 이미 떠 있음)
```bash
pip install strands-decider            # 전용 venv: /data/venvs/strands-decider
HF_HOME=/data/hf-cache strands-decider serve \
  StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --port 8002 --device cuda
```

In [1]:
import sys; sys.path.insert(0, "/home/ubuntu/LLMRouter/OpenSource/workshop")  # ws_common 경로
import json, urllib.request
import ws_common as W

print(json.dumps(json.load(urllib.request.urlopen("http://localhost:8002/health", timeout=5)), indent=1))

{
 "status": "ok",
 "model": "strands-decider-2B-qwen3.5-v1-2610",
 "checkpoint": "StrandsAgents/strands-decider-2B-qwen3.5-v1-2610",
 "base_model": "Qwen/Qwen3.5-2B-Base",
 "num_slots": 24,
 "max_length": 4096,
 "temperature": 1.0545210242263665,
 "device": "cuda",
 "prefix_cache": true,
 "vision": false
}


## 1. 단건 판단 체험 (한국어)

wire format이 같으므로 **지금까지 쓰던 `SGLangDecision` client에 port만 8002로** 주면 됩니다. D1/D2를 한국어로 물어봅니다.

In [2]:
from agent.decisions import SGLangDecision
from agent.prompts import D1_QUESTION, D2_QUESTION
from agent.state import CATEGORIES, D1_CHOICES

strands_dec = SGLangDecision(model="strands-decider", port=8002)   # client 재사용
msg = "지갑을 도둑맞았어요. 카드 정지시켜 주세요."
for q, ch, did in ((D1_QUESTION, D1_CHOICES, "d1"), (D2_QUESTION, CATEGORIES, "d2")):
    r = strands_dec.decide(state_text=msg, question=q, choices=ch, decision_id=did)
    print(f"[{did}] -> {r.label:<16s} confidence {r.confidence:.3f}   {r.latency_ms:,.1f}ms")

[d1] -> in_scope         confidence 0.222   89.7ms
[d2] -> card_security    confidence 0.760   80.4ms


**관찰**: D2(업무 분류)는 한국어에서도 card_security를 높은 confidence로 맞히지만, **D1의 confidence가 0.2~0.3대로 낮게** 나옵니다. 이것이 calibrated confidence의 가치입니다 — 모델이 "이 판단은 자신 없다"를 숫자로 알려주므로, **confidence threshold 미달 시 상위 모델로 escalate하는 cascade**를 안전하게 설계할 수 있습니다. (04에서 본 Qwen3-VL은 과차단 시에도 conf 1.0으로 포화되어 이 설계가 불가능했습니다.)

## 2. 우리 LangGraph workflow에 그대로 꽂기

decision backend는 `.decide()` 서명만 맞으면 교체 가능합니다. **코드 변경 없이 client의 port만 바꿔** 상담 예제 2(해외송금 지연, tool 재시도 loop)를 strands-decider 판단으로 실행합니다.

> 예제 1("카드가 안 긁히는데")이 아닌 예제 2를 쓰는 이유: strands-decider의 D1이 예제 1 질문을 out_of_scope(conf 0.498)로 **오차단**해 workflow가 2 STEP으로 끝납니다. 직접 바꿔서 실행해 보세요 — 아래 품질 실측 표의 D1 0.613이 실제 체감으로 나타나는 사례입니다.


In [3]:
from agent.graph_v2 import build_graph_v2

client, llm = W.make_clients()
graph_strands = build_graph_v2(strands_dec, llm)                     # 판단: strands-decider
graph_sglang  = build_graph_v2(SGLangDecision(port=8001), llm)       # 판단: SGLang Qwen3-VL (비교용)
sc = W.SCENARIOS[2]
print("시나리오:", sc["title"], "|", sc["question"])

시나리오: 시스템 오류 -> tool retry loop | 어제 오후에 보낸 해외송금이 아직 수취인에게 도착하지 않았어요. 송금 번호는 TR-20261026-8841, 금액은 USD 2,000입니다. 확인해 주세요.


In [4]:
res_strands = W.run_verbose(graph_strands, sc["question"], sc["script"], "StrandsDecider")


[StrandsDecider] 고객: 어제 오후에 보낸 해외송금이 아직 수취인에게 도착하지 않았어요. 송금 번호는 TR-20261026-8841, 금액은 USD 2,000입니다. 확인해 주세요.

STEP 1 ▶ guardrail_check — D1 접수 심사: 정상 문의 / 범위 밖 / compliance 위반
      판정[D1] -> in_scope   (89ms, confidence 0.658)

STEP 2 ▶ intent_router — D2 업무 분류: 12개 전문 상담 category 중 선택
      판정[D2] -> transfers   (86ms, confidence 0.877)



STEP 3 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> info_sufficient   (85ms, confidence 0.604)

STEP 4 ▶ tool_step — D3 tool 선택 -> tool 실행 -> D6 결과 검토(성공/재시도/포기)
      판정[D3] -> track_transfer   (85ms, confidence 0.930)
      tool 실행: track_transfer   (0.0ms)
      판정[D6] -> try_alternative   (96ms, confidence 0.706)



STEP 5 ▶ tool_step — D3 tool 선택 -> tool 실행 -> D6 결과 검토(성공/재시도/포기)
      판정[D3] -> transfer_info   (85ms, confidence 0.810)
      tool 실행: transfer_info   (0.0ms)
      판정[D6] -> goal_met   (95ms, confidence 0.739)



STEP 6 ▶ summarize — Sonnet이 tool 결과를 고객 응대용 사실 요약으로 정리
      생성(Sonnet 4.5)   (4,547ms)
      요약(첫 줄): Based on the tool result provided, I cannot extract specific resolution facts ab

STEP 7 ▶ risk_check — D7 리스크 점검: fraud/AML 모니터링 flag 유무
      판정[D7] -> no_risk   (148ms, confidence 0.868)



STEP 8 ▶ escalation_gate — D4 처리 방침: 종결 / 추가 질문 / 상담사 이관
      판정[D4] -> human_handoff   (152ms, confidence 0.140)



STEP 9 ▶ respond — Sonnet이 최종 고객 응답 작성
      생성(Sonnet 4.5)   (5,679ms)
      --- 최종 응답 ---
      안녕하세요,
      
      고객님의 해외송금 건(TR-20261026-8841, USD 2,000)을 확인하고 있습니다. 해외송금은 중개은행 경유 및 수취은행 처리 시간에 따라 영업일 기준 3~5일이 소요될 수 있습니다.
      
      보다 정확한 송금 추적과 현재 진행 상황을 확인하기 위해 전문 상담사에게 사례를 연결해 드렸습니다. 상담사가 송금 경로를 추적하여 1영업일 내에 등록하신 연락처로 상세한 안내를 드릴 예정입니다.
      
      불편을 드려 죄송하며, 신속히 확인하여 도움드리겠습니다.
      
      감사합니다.

>>> [StrandsDecider] 결정 9회 합계 920ms | 생성 10,226ms | e2e 11,155ms


In [5]:
res_sglang = W.run_verbose(graph_sglang, sc["question"], sc["script"], "SGLang")


[SGLang] 고객: 어제 오후에 보낸 해외송금이 아직 수취인에게 도착하지 않았어요. 송금 번호는 TR-20261026-8841, 금액은 USD 2,000입니다. 확인해 주세요.

STEP 1 ▶ guardrail_check — D1 접수 심사: 정상 문의 / 범위 밖 / compliance 위반
      판정[D1] -> in_scope   (37ms, confidence 1.000)

STEP 2 ▶ intent_router — D2 업무 분류: 12개 전문 상담 category 중 선택
      판정[D2] -> transfers   (34ms, confidence 1.000)

STEP 3 ▶ clarify_gate — D5 정보 충분성 판정: 바로 처리 가능한가, 되물어야 하는가
      판정[D5] -> info_sufficient   (33ms, confidence 1.000)

STEP 4 ▶ tool_step — D3 tool 선택 -> tool 실행 -> D6 결과 검토(성공/재시도/포기)
      판정[D3] -> track_transfer   (33ms, confidence 1.000)
      tool 실행: track_transfer   (0.0ms)
      판정[D6] -> try_alternative   (35ms, confidence 1.000)



STEP 5 ▶ tool_step — D3 tool 선택 -> tool 실행 -> D6 결과 검토(성공/재시도/포기)
      판정[D3] -> transfer_info   (33ms, confidence 1.000)
      tool 실행: transfer_info   (0.0ms)
      판정[D6] -> goal_met   (35ms, confidence 0.994)



STEP 6 ▶ summarize — Sonnet이 tool 결과를 고객 응대용 사실 요약으로 정리
      생성(Sonnet 4.5)   (4,676ms)
      요약(첫 줄): Based on the tool result provided, I cannot extract specific resolution facts be

STEP 7 ▶ risk_check — D7 리스크 점검: fraud/AML 모니터링 flag 유무
      판정[D7] -> no_risk   (39ms, confidence 1.000)

STEP 8 ▶ escalation_gate — D4 처리 방침: 종결 / 추가 질문 / 상담사 이관
      판정[D4] -> clarify   (35ms, confidence 0.343)



STEP 9 ▶ respond — Sonnet이 최종 고객 응답 작성
      생성(Sonnet 4.5)   (5,890ms)
      --- 최종 응답 ---
      고객님, 문의해 주셔서 감사합니다.
      
      송금 TR-20261026-8841 건을 확인하기 위해 몇 가지 추가 정보가 필요합니다:
      
      1. **수취인의 은행명과 국가**는 어디인가요?
      2. **송금 신청 시 선택하신 송금 방식**(일반/긴급)은 무엇인가요?
      3. **수취인 계좌번호 또는 SWIFT/IBAN 코드**를 확인해 주시겠어요?
      
      이 정보를 주시면 송금 상태와 예상 도착 시간을 정확히 확인해 드리겠습니다.
      
      해외송금은 통상 1-5 영업일이 소요되며, 수취 은행의 처리 시간에 따라 달라질 수 있습니다.

>>> [SGLang] 결정 9회 합계 315ms | 생성 10,566ms | e2e 10,889ms


In [6]:
W.compare(res_strands, res_sglang)

| 지표 | StrandsDecider | SGLang | 배수(StrandsDecider/SGLang) |
|---|---:|---:|---:|
| 결정 횟수 | 9 | 9 | |
| **결정 latency 합계** | **920ms** | **315ms** | **2.9x** |
| 생성(LLM) latency | 10,226ms | 10,566ms | **1.0x** |
| e2e | 11,155ms | 10,889ms | **1.0x** |

**결정 계층 속도 차이: 2.9x** (e2e 단축 267ms)

**결정별 latency**

| 노드 | 결정 | StrandsDecider | SGLang | 배수 | 판정 label |
|---|---|---:|---:|---:|---|
| guardrail_check | D1 | 89ms | 37ms | **2.4x** | in_scope (동일) |
| intent_router | D2 | 86ms | 34ms | **2.5x** | transfers (동일) |
| clarify_gate | D5 | 85ms | 33ms | **2.6x** | info_sufficient (동일) |
| tool_select | D3 | 85ms | 33ms | **2.6x** | track_transfer (동일) |
| tool_review | D6 | 96ms | 35ms | **2.7x** | try_alternative (동일) |
| tool_select | D3 | 85ms | 33ms | **2.5x** | transfer_info (동일) |
| tool_review | D6 | 95ms | 35ms | **2.7x** | goal_met (동일) |
| risk_check | D7 | 148ms | 39ms | **3.8x** | no_risk (동일) |
| escalation_gate | D4 | 152ms | 35ms | **4.3x** | StrandsDecider=human_handoff / SGLang=clarify |

## 한국어 판단 품질 실측 (3 backend 비교)

이 워크샵의 한국어 160건(객관적 gold)으로 D1/D2를 전수 측정한 결과입니다 (`scripts/95_decider_quality_bench.py`, `results/summary/decider_quality_ko.json`):

| backend | D1 guardrail (3지) | D2 routing strict/lenient (12지) | p50 latency |
|---|---|---|---|
| SGLang Qwen3-VL-8B | **0.913** | **0.629 / 0.707** | 33ms |
| strands-decider 2B | 0.613 | 0.500 / 0.593 | 85ms |
| Laya multilingual 322M | 0.419 | 0.357 / 0.379 | 21ms |

**해석**: 한국어 특화(Laya multilingual)보다 **모델 용량이 품질 상한을 지배**했습니다 (8B > 2B > 0.3B). Laya 측정치는 공식 문서의 zero-shot 수치(0.362)와 일치하며, Laya는 RLCD fine-tuning으로 0.766까지 오르는 **'튜닝 전제' 모델**입니다. 반대로 latency·풋프린트는 작을수록 유리합니다 — 품질/비용/속도의 3축 trade-off를 그대로 보여주는 결과입니다.


## 3. 정리 — 어디에 어울리는 선택인가

| 관점 | strands-decider (2B 전용) | SGLang + Qwen3-VL (서빙 LLM에 endpoint 추가) |
|---|---|---|
| 배포 | 전용 소형 모델 1개 (CPU도 가능, README 기준 ~489ms/질문) | 이미 서빙 중인 LLM/VLM 재사용 (한계비용 ~0) |
| confidence | **calibrated** (낮은 confidence = 진짜 모름) -> cascade 설계 용이 | 포화 경향 (threshold 설계 어려움) |
| 멀티모달 | 텍스트 전용 (vision=false) | 이미지 판단 가능 (05 실습) |
| 생태계 | Strands Agents SDK intervention 공식 예제 | System One wire format 호환 전반 |

- 판정 latency는 이 환경(H100, SGLang과 GPU 공유)에서 warm 기준 **~80ms**로, SGLang(~35ms)보다 느리지만 Bedrock LLM 판단(~640ms) 대비 충분히 빠릅니다.
- 한국어는 2B torso 기준 "쓸 만하지만 확신이 낮은" 수준 — confidence가 그것을 정직하게 드러내므로, **한국어 판단은 낮은 threshold + 상위 모델 fallback** 구성이 자연스럽습니다.